# 6. LangChain Output Parser

- **Output Parser**는 LLM이 생성한 텍스트를 프로그램에서 쓰기 좋은 구조(문자열, 리스트, 딕셔너리, Pydantic 모델 등)로 변환하는 컴포넌트입니다.
- 지금은 많은 LLM이 구조화 출력(JSON Schema, 함수 호출, Structured Output)을 **자체 지원**합니다.
  가능하면 모델의 기본 구조화 기능(`llm.with_structured_output()`)을 먼저 쓰고,
  모델이 지원하지 않거나 추가 검증·후처리가 필요할 때 Output Parser를 사용합니다.

## 학습 목표

1. 주요 Output Parser의 용도를 구분합니다.
2. `StrOutputParser`로 응답을 문자열로 받습니다.
3. `PydanticOutputParser`로 응답을 Python 객체로 받습니다.
4. `CommaSeparatedListOutputParser`로 응답을 리스트로 받습니다.

**기준 버전**: LangChain 1.x, `langchain-openai` 1.x, 모델 `gpt-6-luna`

## 1. 주요 Output Parser (`langchain_core.output_parsers`)

| Parser | 용도 | 비고 |
|---|---|---|
| `StrOutputParser` | 결과를 문자열 그대로 반환 | 가장 단순 |
| `JsonOutputParser` | 텍스트를 JSON(dict)으로 파싱 | 함수 호출 없이 JSON이 필요할 때 |
| `CommaSeparatedListOutputParser` | 쉼표로 구분된 텍스트 → `list[str]` | 간단한 목록 |
| `PydanticOutputParser` | 결과를 Pydantic 모델로 검증·구조화 | 스키마 검증에 강점 |
| `XMLOutputParser` | XML 출력 → dict | XML 형식이 필요할 때 |

> `OutputFixingParser`, `RetryOutputParser`, `EnumOutputParser`, `DatetimeOutputParser` 등은 LangChain 1.x에서 레거시 패키지(`langchain-classic`)로 옮겨졌습니다. 새 코드에서는 `with_structured_output()`을 우선 고려합니다.

이 노트북에서는 `StrOutputParser`, `PydanticOutputParser`, `CommaSeparatedListOutputParser`를 실습합니다.

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키와 모델명을 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai pydantic python-dotenv
```

In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

## 2. StrOutputParser

응답(`AIMessage`)에서 텍스트만 꺼내 **문자열 그대로** 반환합니다.

```text
PromptTemplate → LLM → StrOutputParser
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("한국에서 꼭 먹어 봐야 할 음식 3가지를 추천해 줘")
parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({})
print(isinstance(result, str))   # True: 문자열로 변환됨
print(result)

## 3. PydanticOutputParser

LLM의 답변을 **원하는 형태(JSON 구조)로 강제**하고, 결과를 Python 객체로 변환합니다.

- `parser.get_format_instructions()`: "어떤 형식으로 답해야 파싱이 가능한지"를 설명하는 **출력 형식 안내문**을 자동으로 만들어 줍니다.
- `.partial(format_instructions=...)`: 이 안내문을 프롬프트에 **미리 고정**합니다.

### 3-1. 이름과 나이 추출하기

In [ ]:
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

class Person(BaseModel):
    name: str
    age: int

parser = PydanticOutputParser(pydantic_object=Person)

prompt = ChatPromptTemplate.from_template(
    """다음 문장에서 이름과 나이를 추출하세요.

문장:
{text}

{format_instructions}"""
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({"text": "홍길동은 올해 20살 대학생입니다."})
print(result)
print(type(result))
print(result.name, result.age)

`get_format_instructions()`가 만들어 주는 안내문을 직접 확인해 봅니다.

In [ ]:
print(parser.get_format_instructions())

### 3-2. 이메일 정보 추출하기

`Field(description=...)`에 각 항목의 설명을 쓰면, 형식 안내문에 포함되어 LLM이 더 정확하게 채웁니다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# 1) Pydantic 모델 정의
class EmailInfo(BaseModel):
    sender_name: str = Field(description="보낸 사람의 이름")
    title: str = Field(description="이메일 제목")
    content: str = Field(description="이메일 내용")

# 2) Parser 생성
parser = PydanticOutputParser(pydantic_object=EmailInfo)

# 3) 프롬프트 (출력 형식 안내문 고정)
prompt = PromptTemplate.from_template(
    "{format_instructions}\n\n이메일:\n{email}"
).partial(format_instructions=parser.get_format_instructions())

# 4) 체인 구성
chain = prompt | llm | parser

# 5) 실행
email_text = """보내는 사람: 김철수
제목: 오늘의 회의록
내용: 오늘 회의록을 보내 드립니다. 내용은 첨부 파일을 확인해 주세요.
"""
parsed = chain.invoke({"email": email_text})

print(parsed)
print(parsed.sender_name)
print(parsed.title)
print(parsed.content)

In [ ]:
# with_structured_output(): 같은 결과를 Output Parser 없이 모델의 구조화 출력 기능으로 얻을 수도 있다.

structured_llm = llm.with_structured_output(EmailInfo)
ret = structured_llm.invoke(f"다음 이메일에서 정보를 추출하세요.\n{email_text}")
print(ret)


### 실습

- 입력: "요리 레시피를 생각해 주세요" / 요리: **카레**
- 출력: 재료 목록(`material`)과 만드는 순서(`step`)를 Pydantic 모델로 받아 아래처럼 출력하세요.

```text
재료 :
카레 가루
감자
...

순서 :
1. ...
2. ...
```

In [ ]:
# 여기에 실습 코드를 작성하세요.

#### 정답 예시

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# 1) Pydantic 모델 정의
class Recipe(BaseModel):
    material: list[str] = Field(description="요리 재료(재료명만 간단히)")
    step: list[str] = Field(description="요리 만드는 순서(짧은 문장, 단계별)")

# 2) Parser 생성
parser = PydanticOutputParser(pydantic_object=Recipe)

# 3) 프롬프트
prompt = PromptTemplate.from_template(
    """당신은 요리 전문가입니다.
아래 요리의 레시피를 작성하세요.

{format_instructions}

요리: {dish}
주의:
- material에는 재료명만 넣고, 수량·단위는 생략해도 됩니다.
- step은 5~8단계로 작성하세요."""
).partial(format_instructions=parser.get_format_instructions())

# 4) 체인 구성 및 실행
chain = prompt | llm | parser
parsed = chain.invoke({"dish": "카레"})

# 5) 출력
print("재료 :")
print("\n".join(parsed.material))
print("\n순서 :")
for i, s in enumerate(parsed.step, 1):
    print(f"{i}. {s}")

## 4. CommaSeparatedListOutputParser

쉼표로 구분된 응답을 **파이썬 리스트(`list[str]`)** 로 변환합니다.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1) Parser와 출력 형식 안내문
output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()

# 2) 프롬프트 (형식 안내문은 partial로 고정)
prompt = PromptTemplate.from_template(
    "{subject} {count}개 팀을 보여 줘.\n{format_instructions}"
).partial(format_instructions=format_instructions)

# 3) 체인 구성 및 실행
chain = prompt | llm | output_parser

parsed_output = chain.invoke({"subject": "한국 프로야구", "count": 5})
print(type(parsed_output))
print(parsed_output)